# 실습 1-2. async generator 구성

## 실습 목표

이번 실습은 FastAPI 스트리밍 응답의 기반이 되는 async generator를 **손으로 직접 만들어 보는** 단계입니다. 서버나 LLM은 잠시 잊고 async generator 자체의 모양과 동작 방식을 익힙니다. 이번 실습이 끝나면 실습 1-3, 1-4에서 OpenAI 응답을 FastAPI 응답으로 흘리는 코드가 자연스럽게 읽힙니다.

다음 흐름으로 진행합니다.

- 일반 generator(`def` + `yield`)와 async generator(`async def` + `yield`, `async for`)의 차이를 코드로 구분
- `await asyncio.sleep`으로 지연이 있는 async generator를 만들어 토큰이 흘러오는 모습 재현
- 다른 async generator를 소비해 가공·재방출하는 변환 패턴(스트리밍 파이프라인의 기본형) 학습

## 1. 환경 준비

이번 실습에서 쓰는 `asyncio` 는 파이썬 표준 라이브러리라 별도 설치가 필요 없습니다. 노트북에서 `async/await` 코드를 셀 단위로 실행할 수 있는지만 확인합니다.

In [1]:
import sys, asyncio
print("python:", sys.version.split()[0])
print("asyncio ok:", asyncio is not None)

python: 3.14.3
asyncio ok: True


## 2. 일반 generator 복습

async generator 로 가기 전, **일반 generator** 가 어떤 것이었는지 한 번 짚고 갑니다.

- `def` 함수 안에 `yield` 가 있으면 그 함수는 **generator 함수**.
- 호출하면 값이 바로 나오는 게 아니라 **generator 객체**가 만들어지고, `for` 로 돌릴 때마다 한 조각씩 값이 나옴.
- 즉 "값을 한 번에 만들지 않고, 필요할 때마다 하나씩 흘려보내는 함수".

In [2]:
def count_up(n):
    for i in range(n):
        yield i

gen = count_up(5)
print(type(gen))

for x in gen:
    print("got:", x)

<class 'generator'>
got: 0
got: 1
got: 2
got: 3
got: 4


여기서 핵심은 두 가지입니다.

1. `count_up(5)` 를 호출한 시점에는 아직 아무 값도 만들어지지 않았다는 것.
2. `for` 가 한 바퀴 돌 때마다 `yield` 까지 실행되고 멈춘다는 것.

이 "한 조각씩 흘려보낸다" 는 감이 async generator 에서도 똑같이 유지됩니다. 차이는 **중간에 `await` 가 가능해진다** 는 것 하나입니다.

## 3. async generator 기본형

이제 같은 구조를 비동기로 바꿉니다.

- 함수 정의: `def` → `async def`
- 안에 `yield` 그대로 사용 (이러면 이 함수는 **async generator 함수**가 됨)
- 소비: `for` → `async for`
- 호출은 `await` 가 가능한 환경 (여기서는 노트북 셀이 그 역할을 해줌)

In [3]:
async def count_up_async(n):
    for i in range(n):
        yield i

agen = count_up_async(5)
print(type(agen))

async for x in agen:
    print("got:", x)

<class 'async_generator'>
got: 0
got: 1
got: 2
got: 3
got: 4


동작은 2 단계와 똑같이 0, 1, 2, 3, 4 가 한 번에 한 조각씩 흘러나옵니다. 바뀐 건 함수 앞의 `async`, 소비할 때의 `async for` 뿐입니다.

그런데 그냥 `async` 만 붙인 거면 굳이 왜 쓰는 걸까요. 다음 단계에서 진짜 이유가 나옵니다.

## 4. 지연이 있는 async generator

async generator 의 진짜 가치는 **`yield` 사이에 `await` 를 넣을 수 있다** 는 것입니다.

- 일반 generator 안에서는 `await` 를 쓸 수 없습니다(=네트워크/파일 비동기 I/O 와 자연스럽게 어울리지 못함).
- async generator 안에서는 `await asyncio.sleep(...)`, `await client.something(...)` 같은 비동기 호출을 사이사이 끼워 넣고, 그 결과를 한 조각씩 `yield` 할 수 있습니다.

여기서는 네트워크 대신 `asyncio.sleep` 으로 "토큰이 천천히 도착하는" 상황을 흉내 냅니다. 이게 곧 LLM 스트리밍의 감입니다.

In [4]:
import asyncio, time

async def fake_token_stream(text, delay=0.2):
    # text 를 공백 기준으로 잘라 토큰 흉내
    for token in text.split():
        await asyncio.sleep(delay)  # 네트워크 지연 흉내
        yield token

t0 = time.time()
async for piece in fake_token_stream("async generator 는 토큰 단위로 흐릅니다"):
    print(f"[{time.time()-t0:.2f}s] {piece}")

[0.20s] async
[0.40s] generator
[0.60s] 는
[0.81s] 토큰
[1.01s] 단위로
[1.21s] 흐릅니다


출력의 왼쪽 시간을 보면 토큰이 한 번에 도착하지 않고 0.2 초 간격으로 흘러오는 것이 보입니다. 이게 LLM 스트리밍에서 "토큰이 도착하는 대로 화면에 흘려보내기" 와 정확히 같은 그림입니다.

정리: **`async def` + `yield` + 중간에 `await`**, 이 세 가지가 async generator 의 본체입니다.

## 5. async generator 를 변환·필터하기

스트리밍 파이프라인에서 자주 쓰는 패턴이 있습니다.

- 원본 async generator 가 토큰을 흘림.
- 다른 async generator 가 그것을 `async for` 로 소비하면서, 변환/필터해서 자기도 `yield` 로 다시 흘려보냄.
- 최종 소비자는 변환된 흐름만 본다.

In [5]:
async def upper_only(source):
    # 원본 async generator 를 소비하면서 대문자로 바꿔 다시 흘려보냄
    async for piece in source:
        if not piece:
            continue  # 빈 조각은 거름
        yield piece.upper()

pipeline = upper_only(fake_token_stream("async generator pipeline ok", delay=0.5))

async for piece in pipeline:
    print("->", piece)

-> ASYNC
-> GENERATOR
-> PIPELINE
-> OK


여기서 두 가지 감을 잡으면 됩니다.

- `upper_only` 는 source 가 **무엇이든** async generator 이기만 하면 동작합니다. 즉 source 자리에 LLM 스트리밍을 끼우면 그대로 LLM 응답 변환기가 됩니다.
- `if not piece: continue` 같은 **가드** 가 변환기 안에 들어가는 게 자연스럽습니다. 실제 LLM 스트리밍에서는 빈 chunk 가 끼어 오기 때문에 이 가드가 필수입니다.

# TODO

지금까지 본 패턴을 직접 손에 익히기 위한 과제입니다.

## TODO 1. 리스트를 async generator 로 흘리기

주어진 리스트의 원소를 0.1 초 간격으로 하나씩 `yield` 하는 async generator `list_stream(items, delay=0.1)` 를 작성하고, `async for` 로 소비해서 출력하세요.

In [6]:
import asyncio

async def list_stream(items, delay=0.5):
    # TODO 1: 여기에 작성하세요
    for item in items:
        await asyncio.sleep(delay)  # 지연 흉내
        yield item

async for x in list_stream(["a", "b", "c", "d"]):
    print(x)

a
b
c
d


## TODO 2. 빈 chunk 가드가 들어간 변환 async generator

다음 세 가지를 만족하는 async generator `clean_stream(source)` 를 작성하세요.

1. 원본 async generator `source` 를 `async for` 로 소비한다.
2. 받은 조각이 `None` 또는 빈 문자열이면 건너뛴다.
3. 나머지는 양 끝 공백을 제거한 뒤 `yield` 한다.

그리고 일부러 빈 조각·`None` 이 섞인 source 를 만들어서 `clean_stream` 이 잘 걸러내는지 확인하세요.

In [7]:
import asyncio

async def noisy_source():
    # TODO 2-1: 여기에 작성하세요
    for i in range(10):
        await asyncio.sleep(0.2)  # 지연 흉내
        if i % 3 == 0:
            yield None  # 잡음 흉내
        else:
            yield f"data-{i}"

async def clean_stream(source):
    # TODO 2-2: 여기에 작성하세요
    async for item in source:
        if item is not None:  # 잡음 제거
            yield item

async for x in clean_stream(noisy_source()):
    print(repr(x))

'data-1'
'data-2'
'data-4'
'data-5'
'data-7'
'data-8'


## TODO 3. 토큰 스트림 → 대문자 변환 → 소비 파이프라인

**아무 코드도 미리 주어지지 않은 상태에서** 다음 세 가지를 처음부터 작성하세요.

1. `token_stream(text, delay=0.1)` : 입력 문자열을 공백 기준으로 잘라 `delay` 간격으로 한 토큰씩 `yield` 하는 async generator.
2. `to_upper(source)` : 원본 async generator 를 소비하면서, 빈 조각은 건너뛰고 나머지는 대문자로 변환해 `yield` 하는 async generator.
3. 위 두 개를 연결해서 `to_upper(token_stream("async generator pipeline ok"))` 의 결과를 `async for` 로 출력. 각 줄에 `[경과시간] 토큰` 형태로 찍기.

**검증 포인트**
- 출력이 시간차로 흐르듯 나와야 함 (한 번에 다 찍히면 `await asyncio.sleep` 이 안 끼었다는 뜻).
- 모든 토큰이 대문자여야 함.
- 중간에 빈 토큰이 끼어들어도 출력이 깨지지 않아야 함 (예: 입력에 공백을 일부러 두 칸 줘서 확인).

In [ ]:
# TODO 3: 여기에 작성하세요
async def token_stream(text, delay=0.1):
    for token in text.split():
        await asyncio.sleep(delay)
        yield token

async def to_upper(source):
    async for piece in source:
        if piece is None:           #빈 토큰일시 continue 
            continue
        text = str(piece).strip()
        if not text:
            continue
        yield text.upper()

async def run_pipeline():
    async for token in to_upper(token_stream("async generator pipeline  ok")):
        print(token)

await run_pipeline()

ASYNC
GENERATOR
PIPELINE
OK


## 정리

- `async def` 함수 안에 `yield` 가 있으면 그 함수는 async generator. 호출 결과는 `async for` 로 한 조각씩 받는다.
- async generator 의 핵심 가치는 **`yield` 사이에 `await` 가 가능하다**는 점. 그래서 네트워크/LLM 같은 비동기 응답을 토큰 단위로 흘리는 데 자연스럽게 맞는다.
- `async for` 로 원본을 소비하면서 가공해서 다시 `yield` 하는 "변환 async generator" 패턴은 그대로 LLM 응답 가공기에 적용된다. 빈 chunk 가드 같은 방어 코드는 이 변환 단계에 모아두면 깔끔하다.